# 03 — Model Training & Evaluation

This notebook trains and evaluates **four fake-news detection models**:

| # | Model | Feature Space |
|---|-------|---------------|
| 1 | TF-IDF + Logistic Regression | TF-IDF bag-of-words (up to 50 k features) |
| 2 | SBERT Embeddings + Logistic Regression | First 384 cols of `features.npz` |
| 3 | Hybrid (SBERT + Stylometric) + Logistic Regression | All 392 cols of `features.npz` |
| 4 | Hybrid + XGBoost | All 392 cols of `features.npz` |

After training, all models are evaluated and the best one (highest F1) is saved to `../models/`.

## 1 · Setup — Imports & Global Config

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Allow imports from the project root
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, auc
)
from xgboost import XGBClassifier

# ── Global settings ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Professional plot style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

matplotlib.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# ── Output directories ───────────────────────────────────────────────────────
OUTPUTS_DIR = '../outputs'
MODELS_DIR  = '../models'
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

print('Setup complete.')
print(f'  outputs dir : {os.path.abspath(OUTPUTS_DIR)}')
print(f'  models  dir : {os.path.abspath(MODELS_DIR)}')

## 2 · Load Pre-computed Features

In [ ]:
# ── Load hybrid feature matrix (SBERT + stylometric) ─────────────────────────
features_path = os.path.join(OUTPUTS_DIR, 'features.npz')
data = np.load(features_path, allow_pickle=True)
X = data['X'].astype(np.float32)
y = data['y'].astype(np.int32)

print(f'Hybrid feature matrix : X.shape = {X.shape}')
print(f'Label vector          : y.shape = {y.shape}')
print(f'Class balance         : {np.bincount(y)} (0=Real, 1=Fake)')

# ── Load combined CSV for raw text (needed by TF-IDF model) ──────────────────
csv_path = os.path.join(OUTPUTS_DIR, 'combined_news.csv')
df = pd.read_csv(csv_path)
print(f'\nDataFrame shape : {df.shape}')
print(f'Columns         : {list(df.columns)}')

# Ensure we have a cleaned_text column; fall back gracefully
if 'cleaned_text' not in df.columns:
    if 'text' in df.columns:
        df['cleaned_text'] = df['text'].fillna('')
        print('  Note: using raw "text" column as cleaned_text')
    else:
        raise KeyError('Neither "cleaned_text" nor "text" column found in CSV.')
else:
    df['cleaned_text'] = df['cleaned_text'].fillna('')

texts = df['cleaned_text'].values
print(f'\nText samples loaded   : {len(texts)}')

# Sanity check: lengths must match
assert len(texts) == len(y), (
    f'Mismatch: features.npz has {len(y)} rows but CSV has {len(texts)} rows.'
)
print('Length sanity check   : PASSED')

## 3 · Train / Test Split

We use a **stratified 80/20 split** so both splits preserve the class ratio.
The same index split is applied to hybrid features **and** raw texts.

In [ ]:
# Create indices so we can align both feature spaces to the same split
indices = np.arange(len(y))

idx_train, idx_test = train_test_split(
    indices,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Hybrid features
X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = y[idx_train], y[idx_test]

# Raw text
texts_train = texts[idx_train]
texts_test  = texts[idx_test]

# SBERT-only slice (first 384 columns)
SBERT_DIM = 384
X_sbert_train = X_train[:, :SBERT_DIM]
X_sbert_test  = X_test[:,  :SBERT_DIM]

print(f'Training samples      : {len(idx_train):,}')
print(f'Test     samples      : {len(idx_test):,}')
print(f'Class balance (train) : {np.bincount(y_train)}')
print(f'Class balance (test)  : {np.bincount(y_test)}')
print(f'Hybrid feature dims   : {X_train.shape[1]}')
print(f'SBERT-only dims       : {X_sbert_train.shape[1]}')

## 4 · Helper — Evaluation Function

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob=None):
    """Return a metrics dict and print a formatted report."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    roc  = roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan

    print(f'\n{"-"*60}')
    print(f'  {name}')
    print(f'{"-"*60}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1 Score  : {f1:.4f}')
    if not np.isnan(roc):
        print(f'  ROC-AUC   : {roc:.4f}')
    else:
        print('  ROC-AUC   : N/A')
    print(f'\nClassification Report:')
    print(classification_report(y_true, y_pred,
                                target_names=['Real (0)', 'Fake (1)']))

    return {
        'Model'    : name,
        'Accuracy' : acc,
        'Precision': prec,
        'Recall'   : rec,
        'F1'       : f1,
        'ROC_AUC'  : roc,
    }

results   = []   # list of metric dicts
trained   = {}   # name -> fitted model (or pipeline)
roc_data  = {}   # name -> (fpr, tpr, auc_val)

print('Helper functions ready.')

## 5 · Model 1 — TF-IDF + Logistic Regression

A strong baseline that represents each article as a sparse bag-of-bigrams vector
(up to 50 000 features). No neural embeddings are used.

In [ ]:
print('Training Model 1: TF-IDF + Logistic Regression ...')

# ── Vectorise text ────────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{2,}',
)
X_tfidf_train = tfidf.fit_transform(texts_train)
X_tfidf_test  = tfidf.transform(texts_test)

print(f'  TF-IDF matrix (train): {X_tfidf_train.shape}')
print(f'  TF-IDF matrix (test) : {X_tfidf_test.shape}')

# ── Classifier ────────────────────────────────────────────────────────────────
lr_tfidf = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    solver='saga',
    C=1.0,
    n_jobs=-1,
)
lr_tfidf.fit(X_tfidf_train, y_train)

y_pred_m1 = lr_tfidf.predict(X_tfidf_test)
y_prob_m1 = lr_tfidf.predict_proba(X_tfidf_test)[:, 1]

metrics_m1 = evaluate_model('Model 1 - TF-IDF + LR', y_test, y_pred_m1, y_prob_m1)
results.append(metrics_m1)
trained['Model 1 - TF-IDF + LR'] = (tfidf, lr_tfidf)

fpr, tpr, _ = roc_curve(y_test, y_prob_m1)
roc_data['TF-IDF + LR'] = (fpr, tpr, auc(fpr, tpr))

## 6 · Model 2 — SBERT Embeddings Only + Logistic Regression

Uses only the first **384 dimensions** of `features.npz` (the SBERT sentence
embeddings) and ignores the hand-crafted stylometric features.

In [ ]:
print('Training Model 2: SBERT Embeddings Only + Logistic Regression ...')

lr_sbert = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    solver='lbfgs',
    C=1.0,
    n_jobs=-1,
)
lr_sbert.fit(X_sbert_train, y_train)

y_pred_m2 = lr_sbert.predict(X_sbert_test)
y_prob_m2 = lr_sbert.predict_proba(X_sbert_test)[:, 1]

metrics_m2 = evaluate_model('Model 2 - SBERT + LR', y_test, y_pred_m2, y_prob_m2)
results.append(metrics_m2)
trained['Model 2 - SBERT + LR'] = lr_sbert

fpr, tpr, _ = roc_curve(y_test, y_prob_m2)
roc_data['SBERT + LR'] = (fpr, tpr, auc(fpr, tpr))

## 7 · Model 3 — Hybrid Features (SBERT + Stylometric) + Logistic Regression

Uses the **full 392-dimensional feature vector**: 384 SBERT dims plus 8
hand-crafted stylometric features (word count, avg word length, punctuation
ratios, etc.).

In [ ]:
print('Training Model 3: Hybrid Features + Logistic Regression ...')

lr_hybrid = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    solver='lbfgs',
    C=1.0,
    n_jobs=-1,
)
lr_hybrid.fit(X_train, y_train)

y_pred_m3 = lr_hybrid.predict(X_test)
y_prob_m3 = lr_hybrid.predict_proba(X_test)[:, 1]

metrics_m3 = evaluate_model('Model 3 - Hybrid + LR', y_test, y_pred_m3, y_prob_m3)
results.append(metrics_m3)
trained['Model 3 - Hybrid + LR'] = lr_hybrid

fpr, tpr, _ = roc_curve(y_test, y_prob_m3)
roc_data['Hybrid + LR'] = (fpr, tpr, auc(fpr, tpr))

## 8 · Model 4 — Hybrid Features + XGBoost

Replaces the linear classifier with a gradient-boosted tree ensemble.
XGBoost can capture non-linear feature interactions that logistic regression misses.

In [ ]:
print('Training Model 4: Hybrid Features + XGBoost ...')
print('  (this may take a minute - 200 trees x 392 features)')

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    verbosity=0,
    n_jobs=-1,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_m4 = xgb_model.predict(X_test)
y_prob_m4 = xgb_model.predict_proba(X_test)[:, 1]

metrics_m4 = evaluate_model('Model 4 - Hybrid + XGBoost', y_test, y_pred_m4, y_prob_m4)
results.append(metrics_m4)
trained['Model 4 - Hybrid + XGBoost'] = xgb_model

fpr, tpr, _ = roc_curve(y_test, y_prob_m4)
roc_data['Hybrid + XGBoost'] = (fpr, tpr, auc(fpr, tpr))

## 9 · Results Comparison

In [ ]:
# ── Build comparison DataFrame ────────────────────────────────────────────────
results_df = pd.DataFrame(results)
results_df = results_df.set_index('Model')
results_df = results_df.round(4)

print('\n' + '='*65)
print('  Model Comparison Summary')
print('='*65)
print(results_df.to_string())

# ── Save to CSV ───────────────────────────────────────────────────────────────
comparison_path = os.path.join(OUTPUTS_DIR, 'model_comparison.csv')
results_df.to_csv(comparison_path)
print(f'\nSaved -> {comparison_path}')

# ── Save classification reports to text file ─────────────────────────────────
report_path = os.path.join(OUTPUTS_DIR, 'classification_report.txt')
pred_map = {
    'Model 1 - TF-IDF + LR'     : (y_pred_m1, y_prob_m1),
    'Model 2 - SBERT + LR'      : (y_pred_m2, y_prob_m2),
    'Model 3 - Hybrid + LR'     : (y_pred_m3, y_prob_m3),
    'Model 4 - Hybrid + XGBoost': (y_pred_m4, y_prob_m4),
}

with open(report_path, 'w') as f:
    f.write('Fake News Detection - Classification Reports\n')
    f.write('='*60 + '\n\n')
    for name, (y_pred, y_prob) in pred_map.items():
        f.write(f'{name}\n')
        f.write('-'*60 + '\n')
        f.write(classification_report(
            y_test, y_pred,
            target_names=['Real (0)', 'Fake (1)']
        ))
        roc_val = roc_auc_score(y_test, y_prob)
        f.write(f'ROC-AUC: {roc_val:.4f}\n\n')

print(f'Saved -> {report_path}')

## 10 · Visualisations

### 10.1 Confusion Matrix — Best Model

In [ ]:
# ── Identify best model by F1 ─────────────────────────────────────────────────
best_row = results_df['F1'].idxmax()
best_f1  = results_df.loc[best_row, 'F1']
print(f'Best model : {best_row}  (F1 = {best_f1:.4f})')

best_pred_map = {
    'Model 1 - TF-IDF + LR'     : y_pred_m1,
    'Model 2 - SBERT + LR'      : y_pred_m2,
    'Model 3 - Hybrid + LR'     : y_pred_m3,
    'Model 4 - Hybrid + XGBoost': y_pred_m4,
}
best_y_pred = best_pred_map[best_row]

# ── Plot confusion matrix ─────────────────────────────────────────────────────
cm      = confusion_matrix(y_test, best_y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Raw counts
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Real', 'Fake'],
    yticklabels=['Real', 'Fake'],
    linewidths=0.5, linecolor='white',
    ax=axes[0]
)
axes[0].set_title(f'Confusion Matrix (Counts)\n{best_row}')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalised
sns.heatmap(
    cm_norm, annot=True, fmt='.2%', cmap='Blues',
    xticklabels=['Real', 'Fake'],
    yticklabels=['Real', 'Fake'],
    linewidths=0.5, linecolor='white',
    vmin=0, vmax=1,
    ax=axes[1]
)
axes[1].set_title(f'Confusion Matrix (Normalised)\n{best_row}')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
cm_path = os.path.join(OUTPUTS_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, bbox_inches='tight')
plt.show()
print(f'Saved -> {cm_path}')

### 10.2 ROC Curves — All Models

In [ ]:
# Colour palette for the four curves
PALETTE = [
    '#E74C3C',  # red    - TF-IDF
    '#3498DB',  # blue   - SBERT
    '#2ECC71',  # green  - Hybrid LR
    '#9B59B6',  # purple - XGBoost
]

fig, ax = plt.subplots(figsize=(8, 7))

for (label, (fpr, tpr, auc_val)), color in zip(roc_data.items(), PALETTE):
    ax.plot(
        fpr, tpr,
        lw=2,
        color=color,
        label=f'{label}  (AUC = {auc_val:.4f})'
    )

# Reference diagonal
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random Classifier (AUC = 0.50)')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Receiver Operating Characteristic - All Models')
ax.legend(loc='lower right', framealpha=0.9)

plt.tight_layout()
roc_path = os.path.join(OUTPUTS_DIR, 'roc_curve.png')
plt.savefig(roc_path, bbox_inches='tight')
plt.show()
print(f'Saved -> {roc_path}')

### 10.3 Metric Bar Chart — All Models

In [ ]:
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC']
plot_df = results_df[metrics_to_plot].copy()

# Shorten model names for the axis
short_names = [
    'TF-IDF\n+ LR',
    'SBERT\n+ LR',
    'Hybrid\n+ LR',
    'Hybrid\n+ XGB',
]
plot_df.index = short_names

fig, ax = plt.subplots(figsize=(11, 5))
x     = np.arange(len(plot_df))
width = 0.15

bar_colors = ['#3498DB', '#E67E22', '#2ECC71', '#E74C3C', '#9B59B6']
for i, (metric, color) in enumerate(zip(metrics_to_plot, bar_colors)):
    bars = ax.bar(
        x + i * width,
        plot_df[metric].values,
        width,
        label=metric,
        color=color,
        alpha=0.85,
        edgecolor='white'
    )
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + 0.004,
            f'{h:.3f}',
            ha='center', va='bottom',
            fontsize=7.5, rotation=90
        )

ax.set_xticks(x + width * 2)
ax.set_xticklabels(short_names)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.legend(loc='lower right', ncol=3, framealpha=0.9)
ax.axhline(y=0.9, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)

plt.tight_layout()
bar_path = os.path.join(OUTPUTS_DIR, 'model_comparison_bar.png')
plt.savefig(bar_path, bbox_inches='tight')
plt.show()
print(f'Saved -> {bar_path}')

## 11 · Select & Save Best Model

In [ ]:
# ── Identify best model ───────────────────────────────────────────────────────
best_name    = results_df['F1'].idxmax()
best_metrics = results_df.loc[best_name].to_dict()
best_model_obj = trained[best_name]

print(f'Best model by F1 : {best_name}')
print(f'Metrics          : {best_metrics}')

# Determine feature count used by best model
if 'TF-IDF' in best_name:
    feature_count = X_tfidf_train.shape[1]
    feature_type  = 'TF-IDF sparse'
elif 'SBERT' in best_name and 'Hybrid' not in best_name:
    feature_count = SBERT_DIM
    feature_type  = 'SBERT dense (384-d)'
else:
    feature_count = X_train.shape[1]
    feature_type  = 'Hybrid dense (SBERT + Stylometric)'

# ── Save best model ───────────────────────────────────────────────────────────
model_path = os.path.join(MODELS_DIR, 'fake_news_model.pkl')
joblib.dump(best_model_obj, model_path, compress=3)
print(f'\nModel saved      -> {model_path}')

# ── Save metadata ─────────────────────────────────────────────────────────────
metadata = {
    'model_name'   : best_name,
    'metrics'      : best_metrics,
    'feature_count': feature_count,
    'feature_type' : feature_type,
    'sbert_dim'    : SBERT_DIM,
    'random_state' : RANDOM_STATE,
    'test_size'    : 0.2,
    'train_samples': int(len(idx_train)),
    'test_samples' : int(len(idx_test)),
    'class_labels' : {0: 'Real', 1: 'Fake'},
}

meta_path = os.path.join(MODELS_DIR, 'model_metadata.pkl')
joblib.dump(metadata, meta_path)
print(f'Metadata saved   -> {meta_path}')

# ── Save the TF-IDF vectoriser separately (useful for inference) ───────────────
tfidf_path = os.path.join(MODELS_DIR, 'tfidf_vectorizer.pkl')
joblib.dump(tfidf, tfidf_path, compress=3)
print(f'TF-IDF saved     -> {tfidf_path}')

# ── Save all models dict ──────────────────────────────────────────────────────
all_models = {
    'tfidf_lr'  : trained['Model 1 - TF-IDF + LR'],
    'sbert_lr'  : trained['Model 2 - SBERT + LR'],
    'hybrid_lr' : trained['Model 3 - Hybrid + LR'],
    'hybrid_xgb': trained['Model 4 - Hybrid + XGBoost'],
}
all_models_path = os.path.join(MODELS_DIR, 'all_models.pkl')
joblib.dump(all_models, all_models_path, compress=3)
print(f'All models saved -> {all_models_path}')

## 12 · Summary

In [ ]:
print('\n' + '='*65)
print('  TRAINING COMPLETE - SUMMARY')
print('='*65)
print(f'\n  Best model  : {best_name}')
print(f'  F1  Score   : {best_metrics["F1"]:.4f}')
print(f'  Accuracy    : {best_metrics["Accuracy"]:.4f}')
print(f'  ROC-AUC     : {best_metrics["ROC_AUC"]:.4f}')
print(f'  Features    : {feature_count} ({feature_type})')

print('\n  Output files:')
output_files = [
    os.path.join(OUTPUTS_DIR, 'model_comparison.csv'),
    os.path.join(OUTPUTS_DIR, 'classification_report.txt'),
    os.path.join(OUTPUTS_DIR, 'confusion_matrix.png'),
    os.path.join(OUTPUTS_DIR, 'roc_curve.png'),
    os.path.join(OUTPUTS_DIR, 'model_comparison_bar.png'),
    os.path.join(MODELS_DIR,  'fake_news_model.pkl'),
    os.path.join(MODELS_DIR,  'model_metadata.pkl'),
    os.path.join(MODELS_DIR,  'tfidf_vectorizer.pkl'),
    os.path.join(MODELS_DIR,  'all_models.pkl'),
]
for f in output_files:
    exists = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'    [{exists}] {os.path.abspath(f)}')

print('\n  Next: run 04_evaluate.ipynb for deeper analysis.')
print('='*65)